In [36]:
import pandas as pd
import json
import os

def read_stats(jsonfilename):
    a = {
        "desktop": {
            "unique_calls": 0,
            "conditional_nodes_counts": 0,
            "iflow_chain_counts": 0
        },
        "mobile": {
            "unique_calls": 0,
            "conditional_nodes_counts": 0,
            "iflow_chain_counts": 0
        }
    }
    with open(jsonfilename) as fp:
        d = json.load(fp)
        fp.close()
    
    for url in d:
        for platform in d[url]:
            if platform == 'code':
                continue
            for api in d[url][platform]:
                if len(d[url][platform][api]['iflow']) > 0:
                    a[platform]['iflow_chain_counts'] += 1
                a[platform]['unique_calls'] += 1
    return a


            

### Total unique calls and corresponding iflows discovered

In [22]:
resultfiles = os.listdir('results')
desktop_iflows, mobile_iflows = 0,0
desktop_calls, mobile_calls = 0,0

for f in resultfiles:
    path = os.path.join('results',f)
    a = read_stats(path)
    desktop_calls += a['desktop']['unique_calls']
    mobile_calls += a['mobile']['unique_calls']
    desktop_iflows += a['desktop']['iflow_chain_counts']
    mobile_iflows += a['mobile']['iflow_chain_counts']

print("Desktop calls: ", desktop_calls)
print("Mobile calls: ", mobile_calls)
print("Desktop iflows: ", desktop_iflows)
print("Mobile iflows: ", mobile_iflows)


Desktop calls:  928
Mobile calls:  499
Desktop iflows:  114
Mobile iflows:  35


In [31]:
iflow_success_rate = (desktop_iflows+mobile_iflows) / (desktop_calls + mobile_calls)
print(f'iflow_success_rate: {iflow_success_rate}')

iflow_success_rate: 0.10441485634197617


### Scripts stat in our database

In [46]:
s = {}
resultfiles = os.listdir('results')
for f in resultfiles:
    path = os.path.join('results',f)
    fp_domain = f.split('.json')[0]
    with open(path) as fp:
        d = json.load(fp)

    for url in d:
        if url not in s:
            s[url] = {
                "unique_calls": 0,
                "iflows": 0,
                "fp": [fp_domain]
            } 
        else:
            s[url]['fp'].append(fp_domain)

        for platform in d[url]:
            if platform == 'code':
                continue
            for api in d[url][platform]:
                s[url]['unique_calls'] += 1
                if len(d[url][platform][api]['iflow']) > 0:
                    s[url]['iflows'] += 1

resultfiles = os.listdir('results')

# dataframe from s
df = pd.DataFrame.from_dict(s, orient='index')
# sort by unique_calls
df = df.sort_values('unique_calls', ascending=False)
# sort by the length of fp array



In [45]:
# check which entry in the df has the highest length of fp_domains and print the url of that row
max_fp = df['fp'].apply(lambda x: len(x)).idxmax()
print("URL with the most FPs: ", df.loc[max_fp].name)

URL with the most FPs:  https://www.google-analytics.com/analytics.js
